# FA 5 – Linear Models (Palmer Penguins)

This notebook generates the required reproducible submission files:

- `README.md`
- `R/FA5_Linear_Models.R`
- `report/FA5_Linear_Models.pdf`
- `figures/body_mass_by_species.png`
- `figures/species_sex_interaction.png`
- `data/penguins.csv`

It follows Parts A–G in the supplied assessment. It uses the real Palmer Penguins CSV, removes rows missing `body_mass_g`, `species`, or `sex`, sets Adelie as the reference species and Female as the reference sex, and reports results from the actual data.

**Run the cells from top to bottom.** The setup cell installs R, R Markdown, and LaTeX packages, which may take several minutes.

In [ ]:
# 1. Install R, Pandoc, R packages, and LaTeX support for PDF rendering.
# This cell is intended for Google Colab's Python runtime.
import os, subprocess, textwrap, pathlib, json, zipfile, shutil, sys

subprocess.run("apt-get update -qq", shell=True, check=True)
subprocess.run(
    "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "r-base r-base-dev pandoc texlive-latex-base texlive-latex-recommended "
    "texlive-fonts-recommended texlive-latex-extra texlive-xetex",
    shell=True, check=True
)
subprocess.run(
    ["R", "-q", "-e",
     'install.packages(c("rmarkdown","knitr","ggplot2"), repos="https://cloud.r-project.org")'],
    check=True
)
print("Setup complete.")

In [ ]:
# 2. Create the project, the complete R script, and the R Markdown report template.
from pathlib import Path
import textwrap, os

PROJECT = Path("/content/FA5-Linear-Models")
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / "R").mkdir(parents=True, exist_ok=True)
(PROJECT / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT / "report").mkdir(parents=True, exist_ok=True)

r_script = r"""
# ============================================================
# FA 5 – LINEAR MODELS: Palmer Penguins
# Parts A–G | Reproducible analysis
# Dataset source: https://github.com/allisonhorst/palmerpenguins
# ============================================================

options(width = 100, scipen = 999)
dir.create("data", showWarnings = FALSE, recursive = TRUE)
dir.create("figures", showWarnings = FALSE, recursive = TRUE)
dir.create("report", showWarnings = FALSE, recursive = TRUE)

data_url <- "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
data_file <- file.path("data", "penguins.csv")
if (!file.exists(data_file)) {
  download.file(data_url, data_file, mode = "wb", quiet = TRUE)
}
penguins <- read.csv(data_file, stringsAsFactors = FALSE)

# Part A: data preparation
cat("\\n================ PART A: DATA PREPARATION ================\\n")
cat("Original dimensions:", nrow(penguins), "rows x", ncol(penguins), "columns\\n")
cat("\\nStructure:\\n")
str(penguins)
cat("\\nSummary:\\n")
print(summary(penguins))
cat("\\nMissing values per column:\\n")
print(colSums(is.na(penguins)))
penguins_clean <- subset(
  penguins,
  !is.na(body_mass_g) & !is.na(species) & !is.na(sex)
)
penguins_clean$species <- factor(
  penguins_clean$species,
  levels = c("Adelie", "Chinstrap", "Gentoo")
)
penguins_clean$sex <- factor(
  penguins_clean$sex,
  levels = c("Female", "Male")
)
cat("\\nFinal sample size:", nrow(penguins_clean), "\\n")
cat("Reference species: Adelie; reference sex: Female\\n")
write.csv(penguins_clean, file.path("data", "penguins_clean.csv"), row.names = FALSE)

# Part B: descriptive statistics
cat("\\n================ PART B: EXPLORATORY ANALYSIS ================\\n")
overall_stats <- data.frame(
  N = sum(!is.na(penguins_clean$body_mass_g)),
  Mean = mean(penguins_clean$body_mass_g),
  SD = sd(penguins_clean$body_mass_g),
  Minimum = min(penguins_clean$body_mass_g),
  Maximum = max(penguins_clean$body_mass_g)
)
species_stats <- aggregate(body_mass_g ~ species, penguins_clean,
  function(x) c(N = length(x), Mean = mean(x), SD = sd(x)))
sex_stats <- aggregate(body_mass_g ~ sex, penguins_clean,
  function(x) c(N = length(x), Mean = mean(x), SD = sd(x)))
species_sex_stats <- aggregate(body_mass_g ~ species + sex, penguins_clean,
  function(x) c(N = length(x), Mean = mean(x), SD = sd(x)))
cat("\\nOverall body mass statistics:\\n"); print(overall_stats)
cat("\\nBody mass by species (N, mean, SD):\\n"); print(species_stats)
cat("\\nBody mass by sex (N, mean, SD):\\n"); print(sex_stats)
cat("\\nBody mass by species and sex (N, mean, SD):\\n"); print(species_sex_stats)

# Required figures
png(file.path("figures", "body_mass_by_species.png"),
    width = 1800, height = 1200, res = 180)
boxplot(body_mass_g ~ species, data = penguins_clean,
        xlab = "Penguin Species", ylab = "Body Mass (g)",
        main = "Penguin Body Mass by Species",
        col = c("#8ecae6", "#ffb703", "#90be6d"))
dev.off()

png(file.path("figures", "body_mass_by_sex.png"),
    width = 1800, height = 1200, res = 180)
boxplot(body_mass_g ~ sex, data = penguins_clean,
        xlab = "Sex", ylab = "Body Mass (g)",
        main = "Penguin Body Mass by Sex",
        col = c("#f4a6c1", "#8ecae6"))
dev.off()

# A summary means table used by the report and interaction plot
cell_summary <- aggregate(body_mass_g ~ species + sex, penguins_clean, mean)
names(cell_summary)[3] <- "mean_body_mass_g"
write.csv(cell_summary, "figures/species_sex_means.csv", row.names = FALSE)

png(file.path("figures", "species_sex_interaction.png"),
    width = 1800, height = 1200, res = 180)
interaction.plot(
  x.factor = penguins_clean$species,
  trace.factor = penguins_clean$sex,
  response = penguins_clean$body_mass_g,
  fun = mean, type = "b", pch = c(16, 17), lty = c(1, 2),
  col = c("#d1495b", "#00798c"),
  xlab = "Penguin Species", ylab = "Mean Body Mass (g)",
  trace.label = "Sex",
  main = "Species × Sex Interaction in Mean Body Mass"
)
dev.off()

# Part C: One-Way ANOVA
cat("\\n================ PART C: ONE-WAY ANOVA ================\\n")
oneway_model <- aov(body_mass_g ~ species, data = penguins_clean)
print(summary(oneway_model))
cat("\\nOne-way model coefficients (equivalent factor regression):\\n")
print(summary(lm(body_mass_g ~ species, data = penguins_clean)))

# Part D: dummy-variable regression
cat("\\n================ PART D: DUMMY-VARIABLE REGRESSION ================\\n")
dummy_model <- lm(body_mass_g ~ species, data = penguins_clean)
print(summary(dummy_model))
cat("\\nCoefficient table:\\n")
print(coef(summary(dummy_model)))
cat("\\nReference species is Adelie. Coefficients for Chinstrap and Gentoo are mean differences from Adelie.\\n")

# Part E: Two-Way ANOVA with interaction.
# Type I sequential ANOVA table, consistent with base R's aov() output.
cat("\\n================ PART E: TWO-WAY ANOVA ================\\n")
twoway_model <- aov(body_mass_g ~ species * sex, data = penguins_clean)
twoway_table <- summary(twoway_model)
print(twoway_table)

# Part F: interaction analysis
cat("\\n================ PART F: INTERACTION ANALYSIS ================\\n")
cat("\\nCell means:\\n")
print(cell_summary)
twoway_tab_df <- as.data.frame(twoway_table[[1]])
interaction_row <- grep("species:sex", rownames(twoway_tab_df))
interaction_p <- if (length(interaction_row)) twoway_tab_df$`Pr(>F)`[interaction_row[1]] else NA_real_
cat("\\nSpecies × Sex interaction p-value:", interaction_p, "\\n")
cat("Interpretation guide: non-parallel lines suggest that the male-female body-mass difference changes by species. Use the interaction p-value to assess statistical evidence.\\n")

# Part G: equivalent full regression model
cat("\\n================ PART G: FULL REGRESSION MODEL ================\\n")
full_model <- lm(body_mass_g ~ species * sex, data = penguins_clean)
print(summary(full_model))
cat("\\nANOVA table for full regression model:\\n")
print(anova(full_model))
cat("\\nReference-cell interpretation: intercept = Adelie females' mean; species coefficients compare female groups with Adelie females; sexMale compares Adelie males with Adelie females; interaction coefficients are additional differences in the male-female contrast for the corresponding species.\\n")

# Save core model outputs as plain text for the report/audit trail.
capture.output({
  cat("FA 5 – Linear Models: reproducible R output\\n")
  cat("Final cleaned sample size:", nrow(penguins_clean), "\\n")
  cat("\\nOverall statistics\\n"); print(overall_stats)
  cat("\\nSpecies summaries\\n"); print(species_stats)
  cat("\\nSex summaries\\n"); print(sex_stats)
  cat("\\nSpecies × sex summaries\\n"); print(species_sex_stats)
  cat("\\nOne-Way ANOVA\\n"); print(summary(oneway_model))
  cat("\\nDummy-variable regression\\n"); print(summary(dummy_model))
  cat("\\nTwo-Way ANOVA\\n"); print(summary(twoway_model))
  cat("\\nFull regression model\\n"); print(summary(full_model))
  cat("\\nFull model ANOVA\\n"); print(anova(full_model))
}, file = "report/statistical_output.txt")

# Render the PDF report from the R Markdown template.
if (file.exists("report/FA5_Linear_Models.Rmd")) {
  rmarkdown::render(
    "report/FA5_Linear_Models.Rmd",
    output_file = "FA5_Linear_Models.pdf",
    output_dir = "report",
    knit_root_dir = ".",
    quiet = TRUE,
    envir = new.env(parent = globalenv())
  )
}
cat("\\nFinished. See report/, figures/, data/, and R/ for generated files.\\n")
"""
(PROJECT / "R" / "FA5_Linear_Models.R").write_text(textwrap.dedent(r_script).strip() + "\n", encoding="utf-8")

rmd = r"""
---
title: "FA 5 – Linear Models"
subtitle: "One-Way ANOVA, Dummy-Variable Regression, Two-Way ANOVA, and Interaction Effects"
author: "Group members: [Enter member name(s)]"
date: "`r format(Sys.Date(), '%B %d, %Y')`"
output:
  pdf_document:
    toc: true
    number_sections: true
    latex_engine: xelatex
geometry: margin=0.8in
---

```{r setup, include=FALSE}
knitr::opts_chunk$set(echo=TRUE, message=FALSE, warning=FALSE, fig.align="center",
                      fig.width=7, fig.height=4.8, out.width="90%")
options(width=100, scipen=999)
penguins <- read.csv("data/penguins.csv", stringsAsFactors=FALSE)
penguins_clean <- subset(penguins, !is.na(body_mass_g) & !is.na(species) & !is.na(sex))
penguins_clean$species <- factor(penguins_clean$species, levels=c("Adelie","Chinstrap","Gentoo"))
penguins_clean$sex <- factor(penguins_clean$sex, levels=c("Female","Male"))
oneway_model <- aov(body_mass_g ~ species, data=penguins_clean)
dummy_model <- lm(body_mass_g ~ species, data=penguins_clean)
twoway_model <- aov(body_mass_g ~ species * sex, data=penguins_clean)
full_model <- lm(body_mass_g ~ species * sex, data=penguins_clean)
overall <- data.frame(
  N=nrow(penguins_clean),
  Mean=mean(penguins_clean$body_mass_g),
  SD=sd(penguins_clean$body_mass_g),
  Minimum=min(penguins_clean$body_mass_g),
  Maximum=max(penguins_clean$body_mass_g)
)
species_summary <- do.call(rbind, lapply(split(penguins_clean$body_mass_g, penguins_clean$species),
  function(x) data.frame(N=length(x), Mean=mean(x), SD=sd(x))))
species_summary$Species <- rownames(species_summary)
rownames(species_summary) <- NULL
species_summary <- species_summary[,c("Species","N","Mean","SD")]
sex_summary <- do.call(rbind, lapply(split(penguins_clean$body_mass_g, penguins_clean$sex),
  function(x) data.frame(N=length(x), Mean=mean(x), SD=sd(x))))
sex_summary$Sex <- rownames(sex_summary)
rownames(sex_summary) <- NULL
sex_summary <- sex_summary[,c("Sex","N","Mean","SD")]
cell_summary <- do.call(rbind, lapply(split(penguins_clean, interaction(penguins_clean$species,penguins_clean$sex, drop=TRUE)),
  function(d) data.frame(Species=as.character(d$species[1]), Sex=as.character(d$sex[1]),
                         N=nrow(d), Mean=mean(d$body_mass_g), SD=sd(d$body_mass_g))))
rownames(cell_summary) <- NULL
twoway_tab <- as.data.frame(summary(twoway_model)[[1]])
interaction_index <- grep("species:sex", rownames(twoway_tab))
interaction_p <- if(length(interaction_index)) twoway_tab$`Pr(>F)`[interaction_index[1]] else NA_real_
species_means <- aggregate(body_mass_g ~ species, penguins_clean, mean)
sex_means <- aggregate(body_mass_g ~ sex, penguins_clean, mean)
male_female_means <- reshape(cell_summary[,c("Species","Sex","Mean")], idvar="Species", timevar="Sex", direction="wide")
```

# Research question

Does penguin body mass differ according to species and sex, and does the effect of species on body mass depend on sex?

The response variable is `body_mass_g` (body mass in grams). The categorical explanatory variables are `species` (Adelie, Chinstrap, Gentoo) and `sex` (Female, Male). The significance level is \(\alpha=0.05\).

# Part A – Data Preparation (5 points)

The dataset is the Palmer Penguins data published by the Palmer Station Long Term Ecological Research program and distributed by the `palmerpenguins` project. The CSV was downloaded from the project’s public repository.

```{r data-preparation}
cat("Original dimensions:", nrow(penguins), "rows and", ncol(penguins), "columns\n")
str(penguins)
summary(penguins)
cat("Missing values per column:\n")
colSums(is.na(penguins))
cat("Rows removed for missing body_mass_g, species, or sex:",
    nrow(penguins) - nrow(penguins_clean), "\n")
cat("Final cleaned sample size:", nrow(penguins_clean), "\n")
```

Rows missing body mass, species, or sex were excluded because these fields are needed for the specified analyses. `species` and `sex` were converted to factors. The reference levels for regression interpretation are Adelie and Female.

# Part B – Exploratory Data Analysis (6 points)

## Descriptive statistics

```{r descriptive-statistics}
knitr::kable(overall, digits=2, caption="Overall body mass statistics (grams)")
knitr::kable(species_summary, digits=2, caption="Body mass by species")
knitr::kable(sex_summary, digits=2, caption="Body mass by sex")
knitr::kable(cell_summary, digits=2, caption="Body mass by species and sex")
```

## Visualizations

```{r species-boxplot, fig.cap="Distribution of penguin body mass by species."}
boxplot(body_mass_g ~ species, data=penguins_clean,
        xlab="Penguin Species", ylab="Body Mass (g)",
        main="Penguin Body Mass by Species",
        col=c("#8ecae6","#ffb703","#90be6d"))
```

```{r sex-boxplot, fig.cap="Distribution of penguin body mass by sex."}
boxplot(body_mass_g ~ sex, data=penguins_clean,
        xlab="Sex", ylab="Body Mass (g)", main="Penguin Body Mass by Sex",
        col=c("#f4a6c1","#8ecae6"))
```

**Observed patterns.** The species with the largest sample mean is `r as.character(species_means$species[which.max(species_means$body_mass_g)])`, and the sex with the largest sample mean is `r as.character(sex_means$sex[which.max(sex_means$body_mass_g)])`. The cell-means table and interaction plot show whether the male–female difference appears similar across species. These are descriptive patterns; inferential conclusions are evaluated in Parts C and E.

# Part C – One-Way ANOVA (7 points)

The model is \(Y_{ij}=\mu+\alpha_j+\epsilon_{ij}\), where \(Y\) is body mass and the groups are species.

- \(H_0\): The population mean body mass is equal for Adelie, Chinstrap, and Gentoo penguins.
- \(H_A\): At least one species population mean body mass differs.

```{r one-way-anova}
summary(oneway_model)
```

```{r one-way-interpretation, results='asis'}
tab <- summary(oneway_model)[[1]]
fval <- tab$`F value`[1]
pval <- tab$`Pr(>F)`[1]
df1 <- tab$Df[1]
df2 <- tab$Df[2]
cat(sprintf("The one-way ANOVA gives F(%d, %d) = %.3f, p %s. ",
            df1, df2, fval,
            ifelse(pval < 0.001, "< 0.001", sprintf("= %.3f", pval))))
if (pval < 0.05) {
  cat("Because p < 0.05, reject the null hypothesis. The data provide evidence that mean body mass is not the same across all three species. ANOVA alone does not identify every pair that differs.")
} else {
  cat("Because p >= 0.05, fail to reject the null hypothesis; this analysis does not provide sufficient evidence of different species means at the 5% level.")
}
```

# Part D – Dummy-Variable Regression (5 points)

Adelie is the reference category. R creates two dummy variables:

- \(D_{\text{Chinstrap}}=1\) for Chinstrap and 0 otherwise.
- \(D_{\text{Gentoo}}=1\) for Gentoo and 0 otherwise.

The model is
\[
Y_i=\beta_0+\beta_1D_{\text{Chinstrap},i}
+\beta_2D_{\text{Gentoo},i}+\epsilon_i.
\]

```{r dummy-regression}
summary(dummy_model)
knitr::kable(coef(summary(dummy_model)), digits=3,
             caption="Dummy-variable regression coefficients")
```

The intercept is the estimated mean body mass for Adelie penguins. The Chinstrap coefficient is the estimated difference between the Chinstrap and Adelie means; the Gentoo coefficient is the estimated difference between the Gentoo and Adelie means. Each coefficient is tested against zero relative to the chosen reference group.

```{r dummy-equivalence, results='asis'}
cat("The factor regression and one-way ANOVA use the same grouping information. The overall F-test for species in the one-way ANOVA tests whether the species means are equal; the regression coefficients express the group mean differences relative to Adelie.")
```

# Part E – Two-Way ANOVA (8 points)

The full factorial model is \(Y \sim \text{Species}+\text{Sex}+\text{Species}\times\text{Sex}\). It evaluates the species main effect, sex main effect, and interaction. The table below is the base R sequential ANOVA table for the specified model order `species * sex`.

```{r two-way-anova}
summary(twoway_model)
```

```{r two-way-decisions, results='asis'}
tab <- summary(twoway_model)[[1]]
for (term in c("species","sex","species:sex")) {
  idx <- match(term, rownames(tab))
  if (!is.na(idx)) {
    pv <- tab$`Pr(>F)`[idx]
    cat("\n\n**", term, "**: ")
    cat(sprintf("F = %.3f, p %s. ", tab$`F value`[idx],
                ifelse(pv < 0.001, "< 0.001", sprintf("= %.3f", pv))))
    if (pv < 0.05) {
      cat("Reject the corresponding null hypothesis at alpha = 0.05.")
    } else {
      cat("Fail to reject the corresponding null hypothesis at alpha = 0.05.")
    }
  }
}
```

- **Species main effect:** tests whether mean body mass differs by species, averaging over sex as represented by this model.
- **Sex main effect:** tests whether mean body mass differs between sexes, accounting for the species term.
- **Species × Sex interaction:** tests whether the sex difference in body mass changes across species. If the interaction is significant, interpret main effects with care because the differences are not constant across the other factor.

# Part F – Interaction Analysis (7 points)

```{r interaction-plot, fig.cap="Mean body mass by species with separate lines for female and male penguins."}
interaction.plot(
  x.factor=penguins_clean$species,
  trace.factor=penguins_clean$sex,
  response=penguins_clean$body_mass_g,
  fun=mean, type="b", pch=c(16,17), lty=c(1,2),
  col=c("#d1495b","#00798c"),
  xlab="Penguin Species", ylab="Mean Body Mass (g)",
  trace.label="Sex", main="Species × Sex Interaction in Mean Body Mass"
)
```

**Interpretation.** Parallel profiles would suggest that the male–female difference is approximately constant across species. Non-parallel profiles suggest that the sex difference changes by species, which is the descriptive pattern associated with interaction. The interaction test from Part E determines whether there is statistically significant evidence of this effect at \(\alpha=0.05\).

```{r interaction-summary}
knitr::kable(cell_summary, digits=2, caption="Cell means used to interpret the interaction")
cat(sprintf("\nSpecies × Sex interaction p-value: %s\n",
            ifelse(is.na(interaction_p), "not available",
                   ifelse(interaction_p < 0.001, "< 0.001", sprintf("%.3f", interaction_p)))))
```

In practical terms, an interaction means that the observed difference in body mass between male and female penguins is not necessarily the same for Adelie, Chinstrap, and Gentoo. The plot describes the pattern; the ANOVA p-value supplies the inferential evidence.

# Part G – Full Regression Model (2 points)

The equivalent regression model is
\[
Y_i=\beta_0+\beta_1D_{\text{Chinstrap},i}
+\beta_2D_{\text{Gentoo},i}+\beta_3D_{\text{Male},i}
+\beta_4(D_{\text{Chinstrap},i}D_{\text{Male},i})
+\beta_5(D_{\text{Gentoo},i}D_{\text{Male},i})+\epsilon_i.
\]

```{r full-regression}
summary(full_model)
anova(full_model)
```

With Adelie and Female as reference levels:

- **Intercept:** estimated mean body mass for Adelie females.
- **Species coefficients:** differences between Chinstrap or Gentoo females and Adelie females.
- **Sex coefficient (`sexMale`):** male–female difference among Adelie penguins.
- **Interaction coefficients:** additional changes in the male–female difference for Chinstrap and Gentoo relative to Adelie. Main effects remain in the model when interactions are included.

# Final Interpretation

The research question asks whether body mass differs according to species and sex and whether the species effect depends on sex. The descriptive tables and plots summarize the observed sample patterns, while the One-Way ANOVA tests species differences and the Two-Way ANOVA evaluates species, sex, and their interaction. At the 5% significance level, interpret each p-value against its corresponding null hypothesis. A statistically significant interaction supports the conclusion that the sex difference in body mass varies across species; a non-significant interaction means the data do not provide sufficient evidence for that variation, even if the plotted sample means are not exactly parallel. Statistical significance should be distinguished from the size and practical meaning of the observed differences.

# Reproducibility

The report uses the Palmer Penguins CSV from the `palmerpenguins` GitHub repository. The R script downloads the dataset if it is not already present, records missingness, removes rows missing body mass/species/sex, factors the categorical variables, saves figures, and renders this report. Numerical results are calculated from the dataset when the notebook is run; no simulated observations are used.

**Dataset source:** <https://github.com/allisonhorst/palmerpenguins>
"""
(PROJECT / "report" / "FA5_Linear_Models.Rmd").write_text(textwrap.dedent(rmd).strip() + "\n", encoding="utf-8")

readme = """# FA 5 – Linear Models

## Group members
- [Enter member name]
- [Enter second member, if applicable]

## Research question
Does penguin body mass differ according to species and sex, and does the effect of species on body mass depend on sex?

## Dataset
Palmer Penguins dataset, distributed by the `palmerpenguins` project. The analysis uses `body_mass_g` as the response and `species` and `sex` as categorical explanatory variables.

Source: https://github.com/allisonhorst/palmerpenguins  
CSV: https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv

## Methods
- Data inspection and missing-value handling
- Descriptive statistics by species, sex, and species × sex
- Boxplots and a species × sex interaction plot
- One-Way ANOVA for species
- Dummy-variable regression using Adelie as reference
- Two-Way ANOVA with species, sex, and their interaction
- Full regression model with interaction terms

Significance level: α = 0.05. The script uses the actual Palmer Penguins data; it does not generate simulated observations.

## Files
- `data/penguins.csv`: source dataset
- `data/penguins_clean.csv`: analysis dataset after removing missing values in body mass, species, and sex
- `R/FA5_Linear_Models.R`: complete R script
- `figures/body_mass_by_species.png`: required species boxplot
- `figures/body_mass_by_sex.png`: sex boxplot
- `figures/species_sex_interaction.png`: required interaction plot
- `report/FA5_Linear_Models.pdf`: completed report
- `report/statistical_output.txt`: captured model output

## Reproduce
Run from the project root in an R environment with `rmarkdown`, `knitr`, and `ggplot2` installed. The script downloads `data/penguins.csv` if missing, saves figures, runs all models, and renders the PDF from `report/FA5_Linear_Models.Rmd`.

In Google Colab, run `FA5_Linear_Models_Colab.ipynb` from top to bottom. The notebook installs R and the required dependencies, runs the script, and packages the project into a ZIP file.

## Summary of findings
The PDF report calculates and interprets the observed group means, ANOVA statistics, regression coefficients, and interaction test directly from the dataset at run time. Read the report's Final Interpretation section for conclusions based on the generated results.
"""
(PROJECT / "README.md").write_text(readme, encoding="utf-8")
print("Project files created in", PROJECT)


In [ ]:
# 3. Download the real dataset, run the R analysis, verify outputs, and create a ZIP.
import subprocess, os, zipfile
from pathlib import Path

os.chdir("/content/FA5-Linear-Models")
data_path = Path("data/penguins.csv")
if not data_path.exists():
    subprocess.run(
        ["bash", "-lc",
         "curl -L --fail --silent --show-error "
         "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv "
         "-o data/penguins.csv"],
        check=True
    )

# Run the complete R script from the project root.
result = subprocess.run(["Rscript", "R/FA5_Linear_Models.R"], text=True, capture_output=True)
print(result.stdout[-12000:])
if result.returncode != 0:
    print(result.stderr[-12000:])
    raise RuntimeError("R script failed. Review the error output above.")

# Check required deliverables.
required = [
    "README.md", "data/penguins.csv", "R/FA5_Linear_Models.R",
    "report/FA5_Linear_Models.pdf", "figures/body_mass_by_species.png",
    "figures/species_sex_interaction.png", "report/statistical_output.txt"
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Required outputs missing: {missing}")

zip_path = "/content/FA5-Linear-Models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in Path(".").rglob("*"):
        if path.is_file() and ".Rproj.user" not in str(path):
            zf.write(path, arcname=str(Path("FA5-Linear-Models") / path))
print("\\nAll required outputs generated.")
print("ZIP archive:", zip_path)
print("Files:", *required, sep="\\n - ")

# Optional: trigger download in Colab.
from google.colab import files
files.download(zip_path)
